# ЛИНЕЙКА ЖЕСТКАЯ

## IMPORT

In [1]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder, TrainValidationSplit
from pyspark.sql import SparkSession, Window
from pyspark.ml.feature import VectorAssembler, OneHotEncoder
import json
from pyspark.sql.functions import col,when,median
import os
import shutil
from pyspark.ml.functions import vector_to_array
import pyspark.sql.functions as F

os.environ['HADOOP_HOME'] = "C:\\hadoop" 
# Добавляем путь к bin в системный PATH
os.environ['PATH'] += os.pathsep + "C:\\hadoop\\bin"

## СОЗДАНИЕ СЕССИИ

In [2]:
spark = SparkSession.builder.appName('Logistic_reg')\
    .config("spark.driver.memory", "10g") \
    .config("spark.executor.memory", "10g") \
    .config("spark.driver.maxResultSize", "4g") \
    .getOrCreate()

## ФУНКЦИЯ ПО СБОРКЕ ***КЕТА

In [3]:
def finalize_csv(folder_path, target_name):
    # 1. Проверяем, существует ли папка
    if not os.path.exists(folder_path):
        print(f"Папка {folder_path} не найдена")
        return

    files = os.listdir(folder_path)
    
    # 2. Ищем файл данных (обычно начинается на part- и заканчивается на .csv)
    # Важно: Spark создает скрытые файлы ._SUCCESS и .crc, их игнорируем
    part_files = [f for f in files if f.startswith("part-") and f.endswith(".csv")]
    
    if not part_files:
        print("Файл с данными не найден в папке")
        return
        
    part_file = part_files[0]
    
    # 3. Формируем путь назначения (в ту же папку, где лежала папка-результат)
    parent_dir = os.path.dirname(os.path.abspath(folder_path))
    final_path = os.path.join(parent_dir, target_name)
    
    # 4. Переносим файл (shutil.move надежнее os.rename между разделами/дисками)
    temp_full_path = os.path.join(folder_path, part_file)
    
    if os.path.exists(final_path):
        os.remove(final_path)
        
    shutil.move(temp_full_path, final_path)
    
    # 5. Удаляем всю временную папку целиком (вместе со скрытыми файлами .crc)
    shutil.rmtree(folder_path)
    print(f"Готово! Файл сохранен как: {final_path}")

## LOAD DOTA

In [4]:
# Load training data
train_data_path = '../datasets/joined/train_data.parquet'
valid_data_path = '../datasets/joined/valid_data.parquet'
features_path = '../datasets/joined/columns_list.json'
# Load train data
training = spark.read.parquet(*[train_data_path,valid_data_path])
# Load valid data
# validating = spark.read.parquet(valid_data_path)

## СОЗДАНИЕ СПИСКА ПРИЗНАКОВ

In [5]:
with open(features_path, 'r') as file:
        cols = json.load(file)

# СОЗДАЛ КОЛОНКИ
feature_cols = cols['all']
cat_cols = cols['cat']
numeric_cols = list(set(feature_cols) - set(cat_cols))

## КОСТЫЛЬНО-МЕДИЦИНСКИЕ ВМЕШАТЕЛЬСТВА В ДАННЫЕ

In [6]:

for cat in cat_cols:
        training = training.withColumn(cat,col(cat)+1)

# ВХОДНЫЕ КАТЕГОРИАЛЬНЫЕ КОЛОНКИ
to_encode_in = cat_cols
to_encode_out = [c+"_vec" for c in cat_cols]


## ПРЕПРОЦЕСС + КРОСВАЛ + ГРИДСЕРЕГА

In [7]:

# ОБЪЯВИЛ ЭНКОДЕР И ОБРАБОТАЛ
encoder = OneHotEncoder(inputCols=to_encode_in,
                        outputCols=to_encode_out,
                        handleInvalid='keep')  # Вот это спасет от ошибки)
training_encoded = encoder.fit(training).transform(training)

# ВЕКТОРИЗОВАЛ
assembler_inputs = to_encode_out + numeric_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features")

final_training = assembler.transform(training_encoded) 

# Define the model
lr = LogisticRegression(
    featuresCol='features',
    labelCol='target'
)

# # Create parameter grid
# paramGrid = (ParamGridBuilder()
#              .addGrid(lr.regParam, [0,0.01, 0.1, 0.3])
#              .addGrid(lr.elasticNetParam, [0.01,0.04,0.07])
#              .addGrid(lr.maxIter, [20,50])
#              .build())
# Create parameter grid
paramGrid = (ParamGridBuilder()
             .addGrid(lr.regParam, [0])
             .addGrid(lr.elasticNetParam, [0.01])
             .addGrid(lr.maxIter, [50])
             .build())

# Define evaluator
evaluator = BinaryClassificationEvaluator(
    rawPredictionCol='rawPrediction',
    labelCol='target',
    metricName='areaUnderROC'
)

sample_fraction = 0.3
print(f"Sampling {sample_fraction*100}% of data for grid search...")
training_sample = final_training.sample(False, sample_fraction, seed=42)

# ЗДЕСЬ ЖЕСТКО НЕ УВЕРЕН ЧТО ЭТО НАДО ВООБЩЕ
training_sample.cache()
print(f"Training sample count: {training_sample.count()}")

# CrossValidator with sampling for big data
# For production, you might want to use 3-5 folds
crossval = CrossValidator(
    estimator=lr,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2,  # Adjust based on your cluster
    seed=42
)

print("Starting grid search...")
cvModel = crossval.fit(training_sample)

best_lr_model = cvModel.bestModel
best_reg_param = best_lr_model.getRegParam()
best_elastic_net = best_lr_model.getElasticNetParam()
best_max_iter = best_lr_model.getMaxIter()
sample_predictions = best_lr_model.transform(training_sample)
sample_auc = evaluator.evaluate(sample_predictions)
print(f"Sample AUC: {sample_auc:.4f}")

training_sample.unpersist()

Sampling 30.0% of data for grid search...
Training sample count: 7267492
Starting grid search...
Sample AUC: 0.8374


DataFrame[customer_id: bigint, event_id: bigint, event_dttm: timestamp, event_type_nm: int, event_desc: int, channel_indicator_type: int, channel_indicator_sub_type: int, currency_iso_cd: int, mcc_code: int, pos_cd: int, browser_language: int, battery: int, device_system_version: int, developer_tools: int, phone_voip_call_state: int, web_rdp_connection: int, compromised: int, target: int, os_category: int, tz_category: int, language_category: int, operaton_amt_is_missing: int, operaton_amt_log: double, hour: int, day_of_week: int, is_night: int, mean_operation: double, median_operation: double, user_activity: bigint, was_hacked: int, avg_ops_last_week: double, count_ops_last_week: bigint, median_ops_last_week: double, inter_quantile_range: double, ops_count_2h: bigint, ops_sum_2h: double, max_amt_2h: double, language_category_vec: vector, compromised_vec: vector, mcc_code_vec: vector, hour_vec: vector, is_night_vec: vector, battery_vec: vector, channel_indicator_sub_type_vec: vector, d

## ЗДЕСЬ ВЫВОД МЕТРИК.. И С ВАЛИДАЦИЕЙ ДЕЙСТВИЯ

In [8]:
print(f'ALPHA: {best_reg_param}, L2RATIO: {best_elastic_net}, MAXITER: {best_max_iter}')

ALPHA: 0.0, L2RATIO: 0.01, MAXITER: 50


## ФИТ ЛУЧШЕЙ МОДЕЛИ

In [9]:
# Create new model with best parameters
best_lr = LogisticRegression(
    featuresCol='features',
    labelCol='target',
    regParam=best_reg_param,
    elasticNetParam=best_elastic_net,
    maxIter=best_max_iter
)

# Train on FULL dataset
final_model = best_lr.fit(final_training)
print("Full model training complete!")

Full model training complete!


## ВАЛИДАЦИЯ БРО

## ЗАКАЧКА ТЕСТА

In [10]:
#  Load test data
test_data_path = '../datasets/joined/test_data.parquet'
testing = spark.read.parquet(test_data_path)

testing = testing.withColumn("was_hacked", 
    when(col("was_hacked") >0, 1).cast("byte")       # В остальных случаях пытаемся в число
).fillna(-1, subset=["was_hacked"])                  # Все пустые/невалидные -> -1
for cat in cat_cols:
        testing = testing.withColumn(cat,col(cat)+1)
        
# ОБЪЯВИЛ ЭНКОДЕР И ОБРАБОТАЛ
test_encoded = encoder.fit(training).transform(testing)

final_test = assembler.transform(test_encoded) 



## ЗДЕСЬ ТАНЦЫ С БУБНОМ (АНСАМБЛЬ И ПРОЧИЕ ПРИБЛУДЫ)

### 1. ДЕЛАЕМ ПРЕДСКАЗАНИЯ

In [11]:
# 1. Трансформируем тестовые данные
predictions = best_lr_model.transform(final_test)

### 2. РАСПРОСТРАНЯЕМ ЧЕРЕЗ ОКОНКУ НА Х ЧАСОВ ПО ЮЗЕРУ

In [12]:
# 1. Сначала превращаем вектор предсказаний в массив, чтобы можно было считать числа
predictions = predictions.withColumn("raw_arr", vector_to_array("rawPrediction"))
# Берем значение для класса 1 (фрод)
predictions = predictions.withColumn("raw_1", F.col("raw_arr")[1])

In [ ]:
time_col = F.col("event_dttm").cast("long")
hours_back = 24
seconds_in_hour = 3600
window_24h = Window.partitionBy("customer_id") \
                          .orderBy(time_col) \
                          .rangeBetween(-hours_back * seconds_in_hour, -1)

# 1. Считаем количество операций (самый важный признак для фрода "набегами")
predictions = predictions.withColumn("max_attention_24h", F.stddev("raw_1").over(window_24h))
        

In [20]:
# Заполняем null (для первой операции юзера, где окна еще нет)
predictions = predictions.fillna(0, subset=["max_attention_24h"])

### 3. ДЕЛАЕМ КОММИТ () ИЛИ СЛОЖИТЬ ИЛИ ОБУЧИТЬ НОВУЮ ЛИНЕЙКУ

In [ ]:
predictions = predictions.withColumn("ultima_answer", col("raw_1") - 0.2 * col("max_attention_24h")) # + max (col(max_att))

## ДУРАЦКАЯ РАБОТА С ПАПКАМИ

In [22]:

# 2. Достаем вероятность КЛАССА 1 (фрод) из вектора [p0, p1]
# vector_to_array превращает вектор в массив, [1] берет второй элемент
# y_out = predictions.withColumn("prob_array", vector_to_array("probability")) \
#                    .select(
#                        F.col("event_id"), 
#                        F.col("prob_array")[1].alias("predict")
#                    )

y_out = predictions.select(
                       F.col("event_id"), 
                       F.col("ultima_answer").alias("predict")
                   )

# 3. Сохраняем во временную ПАПКУ
temp_path = '../submit/temp_folder'
y_out.coalesce(1).write.csv(
    temp_path,
    header=True,
    mode='overwrite'
)

# 4. Вызываем твою функцию 
# Первый аргумент - ПАПКА, которую создал Spark
# Второй аргумент - ИМЯ итогового файла
finalize_csv(temp_path, "lr_submit_1.csv")


Готово! Файл сохранен как: e:\antifraud_hak\submit\lr_submit_1.csv
